In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

# Make project root importable when running from notebooks/
project_root = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

load_dotenv()

True

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel, Field

from src.prompts import QUERY_EXPANSION_PROMPT
from src.utils import get_google_model


class Queries(BaseModel):
    """List of queries to search for."""

    queries: list[str] = Field(description="List of relevant queries to search for")

In [3]:
async def query_expansion_agent(query: str) -> list[str]:

    agent = create_agent(
        model=get_google_model(),
        response_format=Queries,  # Auto-selects ProviderStrategy
    )

    result = agent.invoke(
        {"messages": [HumanMessage(content=QUERY_EXPANSION_PROMPT.format(query=query))]}
    )
    return result["structured_response"].queries

In [4]:
result = await query_expansion_agent("machine learning")
result

['machine-learning',
 'deep-learning',
 'scikit-learn',
 'transformer-models',
 'computer-vision']